# AIME Problem Investigation

Simple notebook to:
1. Load an AIME problem and solution
2. Solve with LLM (Opus)
3. Solve with SLM (Haiku)

In [6]:
# Import necessary modules
from datasets import load_dataset
from lossy_compression import model_completion, LLM, SLM, MODEL_ALIAS_MAP
from lossy_compression.model_messaging_wrappers import MATH_SOLVER_SYSTEM
import re

## 1. Load Problem and Solution

In [7]:
# Load AIME dataset
ds = load_dataset("AI-MO/aimo-validation-aime")
dataset = ds['train']

# Select a problem (change index to explore different problems)
PROBLEM_INDEX = 6
SLM = MODEL_ALIAS_MAP["sonnet"]

problem_data = dataset[PROBLEM_INDEX]
problem_text = problem_data['problem']
correct_answer = problem_data['answer']
solution_text = problem_data['solution']
print(f"Problem {PROBLEM_INDEX}:")
print("="*60)
print(problem_text)
print("="*60)
print(f"\nCorrect answer: {correct_answer}")

Problem 6:
Let $x,$ $y,$ and $z$ be positive real numbers satisfying the system of equations:
\begin{align*} \sqrt{2x-xy} + \sqrt{2y-xy} &= 1 \\ \sqrt{2y-yz} + \sqrt{2z-yz} &= \sqrt2 \\ \sqrt{2z-zx} + \sqrt{2x-zx} &= \sqrt3. \end{align*} 
Then $\left[ (1-x)(1-y)(1-z) \right]^2$ can be written as $\frac{m}{n},$ where $m$ and $n$ are relatively prime positive integers. Find $m+n.$

Correct answer: 033


In [8]:
problem_data

{'id': 6,
 'problem': 'Let $x,$ $y,$ and $z$ be positive real numbers satisfying the system of equations:\n\\begin{align*} \\sqrt{2x-xy} + \\sqrt{2y-xy} &= 1 \\\\ \\sqrt{2y-yz} + \\sqrt{2z-yz} &= \\sqrt2 \\\\ \\sqrt{2z-zx} + \\sqrt{2x-zx} &= \\sqrt3. \\end{align*} \nThen $\\left[ (1-x)(1-y)(1-z) \\right]^2$ can be written as $\\frac{m}{n},$ where $m$ and $n$ are relatively prime positive integers. Find $m+n.$',
 'solution': "First, let define a triangle with side lengths $\\sqrt{2x}$, $\\sqrt{2z}$, and $l$, with altitude from $l$'s equal to $\\sqrt{xz}$. $l = \\sqrt{2x - xz} + \\sqrt{2z - xz}$, the left side of one equation in the problem.\nLet  $\\theta$ be angle opposite the side with length $\\sqrt{2x}$. Then the altitude has length $\\sqrt{2z} \\cdot \\sin(\\theta) = \\sqrt{xz}$ and thus $\\sin(\\theta) = \\sqrt{\\frac{x}{2}}$, so $x=2\\sin^2(\\theta)$ and the side length $\\sqrt{2x}$ is equal to $2\\sin(\\theta)$.\nWe can symmetrically apply this to the two other equations/triangl

## 2. Solve with LLM (Opus)

In [9]:
# Solve with LLM
llm_response = model_completion(
    problem_text,
    model=LLM,  # This is Opus by default
    system=MATH_SOLVER_SYSTEM,
    temperature=0.1,
    max_tokens=2000
)

print("LLM (Opus) Solution:")
print("="*60)
print(llm_response)
print("="*60)

# Extract answer from LLM response
llm_answer_match = re.search(r'\\boxed\{([^}]+)\}', llm_response)
llm_answer = llm_answer_match.group(1).strip() if llm_answer_match else "Not found"
print(f"\nLLM Answer: {llm_answer}")
print(f"Correct: {llm_answer == correct_answer}")

LLM (Opus) Solution:
Problem: Find $m+n$ where $[(1-x)(1-y)(1-z)]^2 = \frac{m}{n}$ in lowest terms, given a system with square root equations.

Solution: Let me make substitutions to simplify. For the first equation:
$$\sqrt{2x-xy} + \sqrt{2y-xy} = \sqrt{x(2-y)} + \sqrt{y(2-x)} = 1$$

Let $a = \sqrt{x}$, $b = \sqrt{y}$, $c = \sqrt{z}$. Then:
$$\sqrt{x(2-y)} = a\sqrt{2-b^2}, \quad \sqrt{y(2-x)} = b\sqrt{2-a^2}$$

So the first equation becomes:
$$a\sqrt{2-b^2} + b\sqrt{2-a^2} = 1$$

Similarly, the other equations become:
$$b\sqrt{2-c^2} + c\sqrt{2-b^2} = \sqrt{2}$$
$$c\sqrt{2-a^2} + a\sqrt{2-c^2} = \sqrt{3}$$

Now let $\alpha = \sqrt{2-a^2}$, $\beta = \sqrt{2-b^2}$, $\gamma = \sqrt{2-c^2}$.

Then $a^2 = 2-\alpha^2$, $b^2 = 2-\beta^2$, $c^2 = 2-\gamma^2$.

The system becomes:
$$a\beta + b\alpha = 1$$
$$b\gamma + c\beta = \sqrt{2}$$
$$c\alpha + a\gamma = \sqrt{3}$$

Also, from $a = \sqrt{2-\alpha^2}$ and $\alpha = \sqrt{2-a^2}$, we have the constraint $a^2 + \alpha^2 = 2$ (similarly for $b

## 3. Solve with SLM (Haiku)

In [11]:
# Solve with SLM
slm_response = model_completion(
    problem_text,
    model=SLM,  # This is Haiku by default
    system=MATH_SOLVER_SYSTEM,
    temperature=0.1,
    max_tokens=2000
)

print("SLM (Haiku) Solution:")
print("="*60)
print(slm_response)
print("="*60)

# Extract answer from SLM response
slm_answer_match = re.search(r'\\boxed\{([^}]+)\}', slm_response)
slm_answer = slm_answer_match.group(1).strip() if slm_answer_match else "Not found"
print(f"\nSLM Answer: {slm_answer}")
print(f"Correct: {slm_answer == correct_answer}")

SLM (Haiku) Solution:
Problem: Find m+n where [(1-x)(1-y)(1-z)]² = m/n for positive reals x,y,z satisfying a system of equations.

Solution:
Let's define:
$a = \sqrt{2x-xy} + \sqrt{2y-xy}$
$b = \sqrt{2y-yz} + \sqrt{2z-yz}$
$c = \sqrt{2z-zx} + \sqrt{2x-zx}$

We're given that $a = 1$, $b = \sqrt{2}$, and $c = \sqrt{3}$.

First, I'll simplify the expressions under the square roots:
$\sqrt{2x-xy} = \sqrt{x(2-y)}$
$\sqrt{2y-xy} = \sqrt{y(2-x)}$

Similarly for the other terms. So our equations become:
$\sqrt{x(2-y)} + \sqrt{y(2-x)} = 1$
$\sqrt{y(2-z)} + \sqrt{z(2-y)} = \sqrt{2}$
$\sqrt{z(2-x)} + \sqrt{x(2-z)} = \sqrt{3}$

Squaring the first equation:
$x(2-y) + y(2-x) + 2\sqrt{x(2-y)y(2-x)} = 1$
$2x - xy + 2y - xy + 2\sqrt{xy(2-y)(2-x)} = 1$
$2x + 2y - 2xy + 2\sqrt{xy(2-y)(2-x)} = 1$

Similarly for the other equations:
$2y + 2z - 2yz + 2\sqrt{yz(2-y)(2-z)} = 2$
$2z + 2x - 2zx + 2\sqrt{zx(2-z)(2-x)} = 3$

Let's set $u = 2-x$, $v = 2-y$, $w = 2-z$. Then:
$2(2-u) + 2(2-v) - 2(2-u)(2-v) + 2\sqrt{

## Summary

In [12]:
# Summary comparison
print("Summary:")
print("="*60)
print(f"Correct Answer: {correct_answer}")
print(f"LLM Answer:     {llm_answer} {'✓' if llm_answer == correct_answer else '✗'}")
print(f"SLM Answer:     {slm_answer} {'✓' if slm_answer == correct_answer else '✗'}")

Summary:
Correct Answer: 033
LLM Answer:     33 ✗
SLM Answer:     577 ✗


##  Investigate differences

In [13]:
print(problem_text)
print("------")

print(solution_text)
print("------")

print(slm_response)
print("------")
print(llm_response)


Let $x,$ $y,$ and $z$ be positive real numbers satisfying the system of equations:
\begin{align*} \sqrt{2x-xy} + \sqrt{2y-xy} &= 1 \\ \sqrt{2y-yz} + \sqrt{2z-yz} &= \sqrt2 \\ \sqrt{2z-zx} + \sqrt{2x-zx} &= \sqrt3. \end{align*} 
Then $\left[ (1-x)(1-y)(1-z) \right]^2$ can be written as $\frac{m}{n},$ where $m$ and $n$ are relatively prime positive integers. Find $m+n.$
------
First, let define a triangle with side lengths $\sqrt{2x}$, $\sqrt{2z}$, and $l$, with altitude from $l$'s equal to $\sqrt{xz}$. $l = \sqrt{2x - xz} + \sqrt{2z - xz}$, the left side of one equation in the problem.
Let  $\theta$ be angle opposite the side with length $\sqrt{2x}$. Then the altitude has length $\sqrt{2z} \cdot \sin(\theta) = \sqrt{xz}$ and thus $\sin(\theta) = \sqrt{\frac{x}{2}}$, so $x=2\sin^2(\theta)$ and the side length $\sqrt{2x}$ is equal to $2\sin(\theta)$.
We can symmetrically apply this to the two other equations/triangles. 
By law of sines, we have $\frac{2\sin(\theta)}{\sin(\theta)} = 2R$, w

In [14]:
LLM_prompt =f""" You are a large language model, interacting with a small language model. I want you to help me figure out a minimal set of instructions that will make the small language model solve the following problem:
{problem_text}

Here is the solution:
{solution_text}. 
Here is the answer you gave: {llm_response}

Here is the answer the small language model gave: {slm_response}

Tell me what you think the small language model is doing wrong. Please be extremely concise. I need to be able to give this to the small language model in a prompt.
"""

# prompt LLM :

llm_response_1 = model_completion(
    LLM_prompt,
    model=LLM,  # This is Opus by default
    system= "", #MATH_SOLVER_SYSTEM,
    temperature=0.1,
    max_tokens=2000
)

print(llm_response_1)

The small language model makes an error when substituting $u = 2-x$, $v = 2-y$, $w = 2-z$. This substitution leads to incorrect algebraic manipulations and wrong values for $u$, $v$, $w$.

The correct approach is to make a trigonometric substitution: Let $x = 2\sin^2(\alpha)$, $y = 2\sin^2(\beta)$, $z = 2\sin^2(\gamma)$. This transforms the equations into:
- $\sin(\alpha + \beta) = 1/2$
- $\sin(\beta + \gamma) = \sqrt{2}/2$  
- $\sin(\gamma + \alpha) = \sqrt{3}/2$

Solving gives $\alpha = \pi/8$, $\beta = \pi/24$, $\gamma = 5\pi/24$.

Then $(1-x)(1-y)(1-z) = -\cos(2\alpha)\cdot(-\cos(2\beta))\cdot(-\cos(2\gamma)) = -\cos(\pi/4)\cos(\pi/12)\cos(5\pi/12)$

This evaluates to $\sqrt{2}/8$, so $[(1-x)(1-y)(1-z)]^2 = 1/32$, giving $m+n = 33$.


In [15]:




LLM_prompt_2 =f""" You are a large language model, interacting with a small language model. I want you to help me figure out a minimal set of instructions that will make the small language model solve the following problem:
{problem_text}

Here is the solution:
{solution_text}. 

Here is the answer the small language model gave: {slm_response}

Here is an analysis of the issues with the small language model's answer.

Help me come up with an extremely short set of instructions that will make the small language model solve the problem correctly. be extremely concise. denote the instructions as with a <guidance> tag.
"""
llm_response_2 = model_completion(
    LLM_prompt_2,
    model=LLM,  # This is Opus by default
    system= "", #MATH_SOLVER_SYSTEM,
    temperature=0.1,
    max_tokens=2000
)
print(llm_response_2)


Looking at the small model's errors:

1. It gets lost in algebraic manipulations without a clear strategy
2. It incorrectly guesses values for u, v, w without justification
3. It misses the key insight about trigonometric substitution that makes the problem tractable

The correct solutions all use the fact that expressions like $\sqrt{x}\sqrt{2-y} + \sqrt{y}\sqrt{2-x}$ suggest the substitution $x = 2\sin^2(\alpha)$ or $x = 2\cos^2(\alpha)$, which transforms the equations into sine addition formulas.

<guidance>
Factor out square roots to get $\sqrt{x}\sqrt{2-y} + \sqrt{y}\sqrt{2-x} = 1$. Substitute $x=2\sin^2\alpha$, $y=2\sin^2\beta$, $z=2\sin^2\gamma$. This gives $\sin(\alpha+\beta)=1/2$, $\sin(\beta+\gamma)=\sqrt{2}/2$, $\sin(\gamma+\alpha)=\sqrt{3}/2$. Solve for angles, then compute $(1-x)(1-y)(1-z)=-\cos(2\alpha)\cos(2\beta)\cos(2\gamma)$.
</guidance>


In [16]:
print(MATH_SOLVER_SYSTEM)

You are a math solution assistant.

Output rules (must follow):
- Return only the math solution content (no markdown fences, no chatter, no apologies).
- Start with: Problem: <repeat the problem in one short line>.
- Then a section: Solution: with clear, minimal but correct steps (LaTeX math allowed).
- End with the final NUMERICAL result on its own line as \boxed{...}.
- CRITICAL: The boxed answer must be a single integer (0-999 for AIME problems).
- Do NOT use symbolic expressions like \boxed{(p+q) mod 1000} - compute the actual number!
- Do not include any text after the boxed answer.

Quality rules:
- Reasoning must be mathematically sound and complete enough to justify the answer.
- Arithmetic/algebra must be correct.
- Calculate all expressions to get a final numerical value.
- Keep steps concise; avoid unnecessary commentary.

Example of CORRECT final answer: \boxed{247}
Example of INCORRECT final answer: \boxed{(123 + 124) mod 1000}




In [17]:
guidance = """"Let P(x) = 2x² + ax + b and Q(x) = -2x² + cx + d. Since both pass through (16,54) and (20,53), add the equations P(x) + Q(x) = (a+c)x + (b+d). This linear function passes through (16,108) and (20,106). Solve for b+d."
""" # problem 0 

guidance = "For each element i in {1,...,n}, count pairs (A,B) where i∈A∩B and |A|=|B|. By symmetry, each element contributes equally. If i is fixed in both sets, choose remaining elements: sum over sizes k=0 to n-1 of choosing k elements from n-1 for each set gives Σ(n-1 choose k)². Use Vandermonde: Σ(m choose k)(m choose m-k) = (2m choose m). Get S_n = n·(2n-2 choose n-1). Compute S_2022/S_2021 = (2022·4042·4041)/(2021³) = (4044·4041)/(2021²). Simplify: gcd(4044·4041, 2021²)=1. Find (4044·4041 + 2021²) mod 1000 = 245." # problem 3

guidance_prompt = f"""
Here is the problem:
{problem_text}

here is some guidance for solving the problem:
{guidance}
""" 


SLM_with_guidance = model_completion(
    guidance_prompt,
    model=SLM,  # This is Haiku by default
    system=MATH_SOLVER_SYSTEM,
    temperature=0.1,
    max_tokens=2000
)

print(SLM_with_guidance)

Problem: Find m+n where [(1-x)(1-y)(1-z)]² = m/n for x,y,z satisfying a system of equations.

Solution:
Let's define $a = \sqrt{2x-xy}$, $b = \sqrt{2y-xy}$, $c = \sqrt{2y-yz}$, $d = \sqrt{2z-yz}$, $e = \sqrt{2z-zx}$, and $f = \sqrt{2x-zx}$.

From the given equations:
$a + b = 1$
$c + d = \sqrt{2}$
$e + f = \sqrt{3}$

Also, $a^2 = 2x-xy = x(2-y)$
$b^2 = 2y-xy = y(2-x)$
$c^2 = 2y-yz = y(2-z)$
$d^2 = 2z-yz = z(2-y)$
$e^2 = 2z-zx = z(2-x)$
$f^2 = 2x-zx = x(2-z)$

From $a + b = 1$, we get $a^2 + b^2 + 2ab = 1$
$x(2-y) + y(2-x) + 2ab = 1$
$2x - xy + 2y - xy + 2ab = 1$
$2x + 2y - 2xy + 2ab = 1$

Similarly:
$2y + 2z - 2yz + 2cd = 2$
$2z + 2x - 2zx + 2ef = 3$

Now, $ab = \sqrt{x(2-y)y(2-x)} = \sqrt{xy(2-y)(2-x)}$
$cd = \sqrt{yz(2-z)(2-y)}$
$ef = \sqrt{zx(2-x)(2-z)}$

Substituting:
$2x + 2y - 2xy + 2\sqrt{xy(2-y)(2-x)} = 1$
$2y + 2z - 2yz + 2\sqrt{yz(2-z)(2-y)} = 2$
$2z + 2x - 2zx + 2\sqrt{zx(2-x)(2-z)} = 3$

Let's try $x = y = z = 1/2$:

For the first equation:
$2(1/2) + 2(1/2) - 2(1/2)(1/2) + 

In [ ]:


LLM_prompt__updated =f""" You are a large language model, interacting with a small language model. I want you to help me figure out a minimal set of instructions that will make the small language model solve the following problem:
{problem_text}

Here is the solution:
{solution_text}. 

Here is the answer the small language model gave: {SLM_with_guidance}

Here is an analysis of the issues with the small language model's answer.

Help me come up with an extremely short set of instructions that will make the small language model solve the problem correctly. be extremely concise.
"""
llm_response_3 = model_completion(
    LLM_prompt__updated,
    model=LLM,  # This is Opus by default
    system= "", #MATH_SOLVER_SYSTEM,
    temperature=0.1,
    max_tokens=4000
)
print(llm_response_3)


Looking at the small model's errors:
1. It incorrectly computes $S_n$ as $\sum_{k=0}^n \binom{n}{k}^2 \cdot k$ (wrong formula)
2. It doesn't derive the correct formula $S_n = n \cdot \binom{2n-2}{n-1}$
3. It doesn't simplify the fraction properly
4. It doesn't compute the final answer correctly

Here are minimal instructions:

**Instructions:**
For each element $i \in \{1,2,...,n\}$, count pairs $(A,B)$ where $|A|=|B|$ and $i \in A \cap B$. This equals $\sum_{k=0}^{n-1}\binom{n-1}{k}^2 = \binom{2n-2}{n-1}$ by choosing subsets of the other $n-1$ elements. Thus $S_n = n\binom{2n-2}{n-1}$.

Compute $\frac{S_{2022}}{S_{2021}} = \frac{2022\binom{4042}{2021}}{2021\binom{4040}{2020}} = \frac{2022 \cdot 4042 \cdot 4041}{2021^3} = \frac{4044 \cdot 4041}{2021^2}$.

Since $\gcd(4044 \cdot 4041, 2021^2) = 1$, we have $p = 4044 \cdot 4041$, $q = 2021^2$.

Find $(4044 \cdot 4041 + 2021^2) \bmod 1000 = (44 \cdot 41 + 21^2) \bmod 1000 = 245$.


In [ ]:
"Let P(x) = 2x² + ax + b and Q(x) = -2x² + cx + d. Since both pass through (16,54) and (20,53), add the equations P(x) + Q(x) = (a+c)x + (b+d). This linear function passes through (16,108) and (20,106). Solve for b+d."


"Since P(x) = 2x² + ax + b and Q(x) = -2x² + cx + d both pass through (16,54) and (20,53), we have P(16) = 54, Q(16) = 54, P(20) = 53, Q(20) = 53. Add P(x) + Q(x) = (a+c)x + (b+d). This linear function passes through (16,108) and (20,106). Find b+d."


'Since P(x) = 2x² + ax + b and Q(x) = -2x² + cx + d both pass through (16,54) and (20,53), we have P(16) = 54, Q(16) = 54, P(20) = 53, Q(20) = 53. Add P(x) + Q(x) = (a+c)x + (b+d). This linear function passes through (16,108) and (20,106). Find b+d.'